In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pycocotools opencv-python-headless

import os, cv2, json, shutil, random, glob
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready


In [ ]:
BASE = '/content/drive/MyDrive/FYP/dataset'

# 26 classes outdoor dataset
OUTDOOR_TRAIN = f'{BASE}/26classes/train'
OUTDOOR_VAL   = f'{BASE}/26classes/valid'
OUTDOOR_TEST  = f'{BASE}/26classes/test'

# Indoor DatasetNinja
INDOOR_TRAIN  = f'{BASE}/indoor-objects-detection-DatasetNinja/train'
INDOOR_VAL    = f'{BASE}/indoor-objects-detection-DatasetNinja/valid'
INDOOR_TEST   = f'{BASE}/indoor-objects-detection-DatasetNinja/test'

# MOTS
MOTS_IMAGES   = f'{BASE}/MOTSChallenge/train/images'
MOTS_ANN_TXT  = f'{BASE}/MOTSChallenge/train/instances_txt'

# Output
OUTPUT = '/content/drive/MyDrive/FYP/fyp-preprocessed'

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT}/{split}/images', exist_ok=True)
    os.makedirs(f'{OUTPUT}/{split}/labels', exist_ok=True)

print("Paths set. Checking they exist...")
for path in [OUTDOOR_TRAIN, INDOOR_TRAIN, MOTS_IMAGES, MOTS_ANN_TXT]:
    exists = os.path.exists(path)
    print(f"  {'✅' if exists else '❌'} {path}")

Paths set. Checking they exist...
  ✅ /content/drive/MyDrive/FYP/dataset/26classes/train
  ✅ /content/drive/MyDrive/FYP/dataset/indoor-objects-detection-DatasetNinja/train
  ✅ /content/drive/MyDrive/FYP/dataset/MOTSChallenge/train/images
  ✅ /content/drive/MyDrive/FYP/dataset/MOTSChallenge/train/instances_txt


In [ ]:
# First let's check what's actually in the 26classes train folder
train_files = os.listdir(OUTDOOR_TRAIN)
imgs = [f for f in train_files if f.endswith(('.jpg', '.png'))]
txts = [f for f in train_files if f.endswith('.txt')]
jsons = [f for f in train_files if f.endswith('.json')]

print(f"Images: {len(imgs)}")
print(f"TXT files: {len(txts)}")
print(f"JSON files: {len(jsons)}")
print(f"\nSample files: {train_files[:10]}")

# Read a sample txt if exists
if txts:
    sample_txt = os.path.join(OUTDOOR_TRAIN, txts[0])
    with open(sample_txt) as f:
        print(f"\nSample label ({txts[0]}):")
        print(f.read()[:300])

Images: 32520
TXT files: 0
JSON files: 1

Sample files: ['traffic-light-68-_jpg.rf.25274b68ef2b258601e2b6c08b314bad.jpg', 'traffic-light-703-_jpg.rf.d6f926d7c9a73a921a6bbc6551e0aeee.jpg', 'traffic-light-86-_jpg.rf.8b2cb7542d5c9682101af5155d644896.jpg', 'traffic-light-650-_jpg.rf.af5d6018460b8039e7b42e5aacc78a61.jpg', 'traffic-light-68-_jpg.rf.cb4c8d352a2bb8f81ce077c7e7caf72c.jpg', 'traffic-light-80-_jpg.rf.1d8bf9ac550f5f999c110399cf9837f5.jpg', 'traffic-light-68-_jpg.rf.a897f27d9445a4493c781a1f53622fc9.jpg', 'traffic-light-816-_jpg.rf.3bacc19b51541da71b8fd4167f861cd7.jpg', 'traffic-light-815-_jpg.rf.b7a09f94290eb4763af58f768f3aa478.jpg', 'traffic-light-691-_jpg.rf.63fc0fef4b45371632bb0804947b7eab.jpg']


In [ ]:
NAVIGATION_CLASSES = {
    'person':        0,
    'bicycle':       1,
    'car':           2,
    'motorcycle':    3,
    'bus':           4,
    'truck':         5,
    'traffic light': 6,
    'stop sign':     7,
    'bench':         8,
    'chair':         9,
    'couch':         10,
    'table':         11,
    'door':          12,
    'window':        13,
    'cabinet':       14,
    'pole':          15,
    'stairs':        16,
    'dog':           17,
    'cat':           18,
    'fire hydrant':  19,
}

print(f"Navigation classes: {len(NAVIGATION_CLASSES)}")
for idx, name in sorted((v,k) for k,v in NAVIGATION_CLASSES.items()):
    print(f"  {idx}: {name}")

Navigation classes: 20
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: bus
  5: truck
  6: traffic light
  7: stop sign
  8: bench
  9: chair
  10: couch
  11: table
  12: door
  13: window
  14: cabinet
  15: pole
  16: stairs
  17: dog
  18: cat
  19: fire hydrant


In [ ]:
def is_valid_image(img_path):
    """
    Filter out:
    - Corrupted images
    - Too dark (brightness < 20)
    - Too small (< 96px)
    - Too blurry
    - Studio/white background shots (like gun image)
    """
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            return False, "corrupted"

        h, w = img.shape[:2]
        if h < 96 or w < 96:
            return False, f"too small {w}x{h}"

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness = np.mean(gray)

        if brightness < 20:
            return False, f"too dark {brightness:.1f}"

        blur = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur < 5:
            return False, f"too blurry {blur:.1f}"

        # Filter studio white background shots
        white_ratio = np.sum(gray > 230) / (h * w)
        if white_ratio > 0.55:
            return False, f"white background {white_ratio:.2f}"

        return True, "ok"
    except Exception as e:
        return False, str(e)


def resize_pad(img, size=320):
    """Resize with letterbox padding to square"""
    h, w = img.shape[:2]
    scale = size / max(h, w)
    nh, nw = int(h * scale), int(w * scale)
    resized = cv2.resize(img, (nw, nh))
    ph, pw = size - nh, size - nw
    return cv2.copyMakeBorder(
        resized, ph//2, ph-ph//2, pw//2, pw-pw//2,
        cv2.BORDER_CONSTANT, value=(114,114,114)
    )

print("Filter functions ready")

Filter functions ready


In [ ]:
def process_indoor(split_dir, output_split):
    """
    Converts Supervisely JSON annotations to YOLO format
    split_dir has img/ and ann/ subfolders
    """
    indoor_to_nav = {
        'door': 'door',
        'openedDoor': 'door',
        'window': 'window',
        'chair': 'chair',
        'table': 'table',
        'cabinet': 'cabinet',
        'couch': 'couch',
        'pole': 'pole',
        # cabinetDoor and refrigeratorDoor excluded
    }

    img_dir = Path(split_dir) / 'img'
    ann_dir = Path(split_dir) / 'ann'

    if not img_dir.exists():
        print(f"  ❌ img folder not found: {img_dir}")
        return 0

    img_files = list(img_dir.glob('*.png')) + list(img_dir.glob('*.jpg'))
    saved = skipped = 0

    for img_path in img_files:
        ann_path = ann_dir / (img_path.name + '.json')
        if not ann_path.exists():
            skipped += 1
            continue

        valid, reason = is_valid_image(img_path)
        if not valid:
            skipped += 1
            continue

        with open(ann_path) as f:
            ann = json.load(f)

        iw = ann['size']['width']
        ih = ann['size']['height']
        labels = []

        for obj in ann['objects']:
            title = obj['classTitle']
            if title not in indoor_to_nav:
                continue
            nav = indoor_to_nav[title]
            if nav not in NAVIGATION_CLASSES:
                continue
            cls_id = NAVIGATION_CLASSES[nav]

            pts = obj['points']['exterior']
            x1, y1 = pts[0]
            x2, y2 = pts[1]

            cx = ((x1+x2)/2) / iw
            cy = ((y1+y2)/2) / ih
            bw = abs(x2-x1) / iw
            bh = abs(y2-y1) / ih

            if bw < 0.01 or bh < 0.01:
                continue

            cx = min(max(cx,0),1)
            cy = min(max(cy,0),1)
            bw = min(bw,1)
            bh = min(bh,1)

            labels.append(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if not labels:
            skipped += 1
            continue

        img = cv2.imread(str(img_path))
        img_r = resize_pad(img)

        fname = f"indoor_{img_path.stem}.jpg"
        cv2.imwrite(f"{OUTPUT}/{output_split}/images/{fname}", img_r)
        with open(f"{OUTPUT}/{output_split}/labels/{fname.replace('.jpg','.txt')}", 'w') as f:
            f.write('\n'.join(labels))

        saved += 1

    print(f"  {output_split}: saved={saved}, skipped={skipped}")
    return saved


print("Processing Indoor dataset...")
t = process_indoor(INDOOR_TRAIN, 'train')
v = process_indoor(INDOOR_VAL,   'val')
e = process_indoor(INDOOR_TEST,  'test')
print(f"Indoor total: {t+v+e}")

Processing Indoor dataset...
